# PlanDiffusion 训练流程

完整训练管道，包含五个阶段：
1. 数据加载（本地 / HuggingFace Hub）
2. Tokens 自回归预训练（GPT-2）
3. 编码器 + 解码器 SFT 微调（BERT + GPT-2）
4. 扩散模型图结构掩码预训练（graph_only）
5. 文本编码 + 图结构条件全量扩散训练（text+graph）

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from functools import partial

# 加载词表常量
_vocab      = json.load(open('data/processed/type_combo_vocab.json', encoding='utf-8'))
PAD_ID      = 0
BOS_ID      = _vocab['BOS_ID']
EOS_ID      = _vocab['EOS_ID']
VOCAB_SIZE  = _vocab['VOCAB_SIZE']
NODE_OFFSET = _vocab['NODE_OFFSET']
N_TYPES     = _vocab['N_TYPES']

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = DEVICE.type == 'cuda'
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)   # 全局共用
print(f'device: {DEVICE}  use_amp: {use_amp}')
print(f'VOCAB_SIZE={VOCAB_SIZE}  BOS={BOS_ID}  EOS={EOS_ID}  NODE_OFFSET={NODE_OFFSET}')

# ── BERT 路径：默认从 HuggingFace 官方加载 ────────────────────
BERT_SOURCE = 'hf'    # 'hf' | 'local'
BERT_PATH   = 'bert-base-chinese'      if BERT_SOURCE == 'hf' \
         else 'models/bert-base-chinese'
print(f'BERT 来源: {BERT_SOURCE}  ({BERT_PATH})')

# GPT-2 配置（全局共用）
GPT2_CFG = dict(
    vocab_size   = VOCAB_SIZE,
    n_embd       = 384,
    n_layer      = 8,
    n_head       = 6,
    n_positions  = 256,
    bos_token_id = BOS_ID,
    eos_token_id = EOS_ID,
    pad_token_id = PAD_ID,
    resid_pdrop  = 0.1,
    embd_pdrop   = 0.1,
    attn_pdrop   = 0.1,
)

NPZ_PATH   = 'data/processed/graph_tokens_combo_5w.npz'
JSONL_PATH = 'data/jsonl/mapped_type_data_zh.jsonl'

## 1. 数据加载

In [ ]:
from torch.utils.data import Subset
import random

# ══════════════════════════════════════════════════════════════
DATA_SOURCE = 'hf'      # 'hf' | 'local'
SUBSET_SIZE = 10000     # 取 N 条；None = 使用全量数据
# ══════════════════════════════════════════════════════════════

from core.dataset import (
    LocalGraphTokenDataset, LocalGraphTextDataset,
    HFGraphTokenDataset, HFGraphTextDataset,
    pretrain_collate, sft_collate,
)

if DATA_SOURCE == 'hf':
    pretrain_ds = HFGraphTokenDataset()
    sft_ds      = HFGraphTextDataset(BERT_PATH, max_text_len=64)
else:
    pretrain_ds = LocalGraphTokenDataset(NPZ_PATH)
    sft_ds      = LocalGraphTextDataset(NPZ_PATH, JSONL_PATH, BERT_PATH, max_text_len=64)

def subset(ds, n, seed=42):
    if n is None or n >= len(ds):
        return ds
    random.seed(seed)
    idx = random.sample(range(len(ds)), n)
    return Subset(ds, idx)

pretrain_ds = subset(pretrain_ds, SUBSET_SIZE)
sft_ds      = subset(sft_ds,      SUBSET_SIZE)

print(f'数据源: {DATA_SOURCE}  SUBSET_SIZE: {SUBSET_SIZE}')
print(f'pretrain_ds: {len(pretrain_ds)} 条')
print(f'sft_ds:      {len(sft_ds)} 条')

## 2. Tokens 自回归预训练（GPT-2）

只用图 token 序列，不需要文本。先训好 GPT-2 再做后续 SFT。

In [ ]:
from core.model import GraphModel

PRETRAIN_BATCH  = 32
PRETRAIN_EPOCHS = 100
PRETRAIN_LR     = 6e-4
PRETRAIN_SAVE   = 'checkpoints/pretrain'
os.makedirs(PRETRAIN_SAVE, exist_ok=True)

pretrain_loader = DataLoader(
    pretrain_ds,
    batch_size=PRETRAIN_BATCH,
    shuffle=True,
    collate_fn=partial(pretrain_collate, pad_id=PAD_ID),
    drop_last=True,
    num_workers=0,
)

pretrain_model = GraphModel(GPT2_CFG).to(DEVICE)
n_params = sum(p.numel() for p in pretrain_model.parameters())
print(f'GraphModel 参数量: {n_params/1e6:.1f}M')

pretrain_opt = AdamW(pretrain_model.parameters(), lr=PRETRAIN_LR,
                     weight_decay=0.1, betas=(0.9, 0.95))
total_steps  = PRETRAIN_EPOCHS * len(pretrain_loader)
pretrain_sch = CosineAnnealingLR(pretrain_opt, T_max=total_steps, eta_min=PRETRAIN_LR * 0.1)
print(f'每 epoch steps: {len(pretrain_loader)}  总 steps: {total_steps}')

In [ ]:
best_pretrain_loss = float('inf')
global_step        = 0
use_amp            = DEVICE.type == 'cuda'
scaler             = torch.amp.GradScaler('cuda', enabled=use_amp)

for epoch in range(PRETRAIN_EPOCHS):
    pretrain_model.train()
    epoch_loss = 0.0

    for token_ids, token_mask in pretrain_loader:
        token_ids  = token_ids.to(DEVICE)
        token_mask = token_mask.to(DEVICE)
        x, y, m   = token_ids[:, :-1], token_ids[:, 1:], token_mask[:, :-1]

        with torch.amp.autocast('cuda', enabled=use_amp):
            out  = pretrain_model(x, m)
            loss = loss_fn(out.logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

        pretrain_opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(pretrain_opt)
        nn.utils.clip_grad_norm_(pretrain_model.parameters(), 1.0)
        scaler.step(pretrain_opt)
        scaler.update()
        pretrain_sch.step()

        epoch_loss  += loss.item()
        global_step += 1
        if global_step % 200 == 0:
            print(f'epoch {epoch+1:3d}  step {global_step:5d}  '
                  f'loss {loss.item():.4f}  lr {pretrain_sch.get_last_lr()[0]:.2e}  '
                  f'scale {scaler.get_scale():.0f}')

    avg = epoch_loss / len(pretrain_loader)
    print(f'=== epoch {epoch+1} avg_loss={avg:.4f} ===')
    if avg < best_pretrain_loss:
        best_pretrain_loss = avg
        torch.save(pretrain_model.state_dict(), f'{PRETRAIN_SAVE}/best.pt')
        print(f'  -> saved  best_loss={best_pretrain_loss:.4f}')

## 3. 编码器 + 解码器 SFT 微调（BERT + GPT-2）

加载预训练 GPT-2 权重，接入 BERT 编码器做 cross-attention，只解冻 BERT 最后两层。

In [ ]:
from core.model import TextConditionedGraphModel
from core.train import freeze_bert_except_last_n

SFT_BATCH  = 32
SFT_EPOCHS = 100   # 正式训练改这里
SFT_LR     = 2e-4
SFT_SAVE   = 'checkpoints/sft'
os.makedirs(SFT_SAVE, exist_ok=True)

sft_loader = DataLoader(
    sft_ds,
    batch_size=SFT_BATCH,
    shuffle=True,
    collate_fn=partial(sft_collate, pad_id=PAD_ID),
    drop_last=True,
    num_workers=0,
)

sft_model = TextConditionedGraphModel(
    bert_path=BERT_PATH,
    gpt2_cfg=GPT2_CFG,
    pretrained_gpt2_path=f'{PRETRAIN_SAVE}/best.pt',  # 加载预训练权重
).to(DEVICE)

freeze_bert_except_last_n(sft_model, n=2)

sft_opt = AdamW(
    filter(lambda p: p.requires_grad, sft_model.parameters()),
    lr=SFT_LR, weight_decay=0.1, betas=(0.9, 0.95),
)
sft_total = SFT_EPOCHS * len(sft_loader)
sft_sch   = CosineAnnealingLR(sft_opt, T_max=sft_total, eta_min=SFT_LR * 0.1)
print(f'每 epoch steps: {len(sft_loader)}  总 steps: {sft_total}')

In [ ]:
best_sft_loss = float('inf')
global_step   = 0
use_amp       = DEVICE.type == 'cuda'
scaler        = torch.amp.GradScaler('cuda', enabled=use_amp)

for epoch in range(SFT_EPOCHS):
    sft_model.train()
    epoch_loss = 0.0

    for token_ids, token_mask, text_ids, text_mask in sft_loader:
        token_ids  = token_ids.to(DEVICE)
        token_mask = token_mask.to(DEVICE)
        text_ids   = text_ids.to(DEVICE)
        text_mask  = text_mask.to(DEVICE)
        x, y, m   = token_ids[:, :-1], token_ids[:, 1:], token_mask[:, :-1]

        with torch.amp.autocast('cuda', enabled=use_amp):
            out  = sft_model(x, m, text_ids, text_mask)
            loss = loss_fn(out.logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

        sft_opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(sft_opt)
        nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
        scaler.step(sft_opt)
        scaler.update()
        sft_sch.step()

        epoch_loss  += loss.item()
        global_step += 1
        if global_step % 200 == 0:
            print(f'epoch {epoch+1:3d}  step {global_step:5d}  '
                  f'loss {loss.item():.4f}  lr {sft_sch.get_last_lr()[0]:.2e}  '
                  f'scale {scaler.get_scale():.0f}')

    avg = epoch_loss / len(sft_loader)
    print(f'=== epoch {epoch+1} avg_loss={avg:.4f} ===')
    if avg < best_sft_loss:
        best_sft_loss = avg
        torch.save(sft_model.state_dict(), f'{SFT_SAVE}/best.pt')
        print(f'  -> saved  best_loss={best_sft_loss:.4f}')

## 4. 扩散模型图结构掩码预训练（graph_only）

只用邻接矩阵 + 节点掩码作为条件，不需要文本。让模型先学会图拓扑约束下的坐标生成。

In [ ]:
from node_diffusion.model     import NodeDiffusionTransformer
from node_diffusion.diffusion import GaussianDiffusion
from node_diffusion.dataset   import NodeTextDataset, HFNodeTextDataset

DIFF_BATCH       = 64
DIFF_TOTAL_STEPS = 200000
DIFF_LR          = 1e-4
DIFF_SAVE        = 'checkpoints/node_diffusion_graph_only'
DIFF_LOG         = 100
DIFF_SAVE_EVERY  = 10000
os.makedirs(DIFF_SAVE, exist_ok=True)

if DATA_SOURCE == 'hf':
    graph_only_ds = HFNodeTextDataset(BERT_PATH, max_text_len=64)
else:
    graph_only_ds = NodeTextDataset(NPZ_PATH, JSONL_PATH, BERT_PATH, max_text_len=64)

graph_only_ds = subset(graph_only_ds, SUBSET_SIZE)  # 复用上面定义的 subset 函数

graph_only_loader = DataLoader(graph_only_ds, batch_size=DIFF_BATCH,
                               shuffle=True, num_workers=0, drop_last=True)

def infinite(loader):
    while True:
        yield from loader

graph_only_iter = infinite(graph_only_loader)

diff_model_go = NodeDiffusionTransformer(
    model_channels=256, num_layers=6, num_heads=4
).to(DEVICE)

diffusion   = GaussianDiffusion(timesteps=1000)
diff_opt_go = AdamW(diff_model_go.parameters(), lr=DIFF_LR, weight_decay=1e-4)
print(f'数据源: {DATA_SOURCE}  扩散训练样本数: {len(graph_only_ds)}')

In [ ]:
diff_model_go.train()
running_loss  = running_rmse = 0.0
scaler_go     = torch.amp.GradScaler('cuda', enabled=use_amp)

for step in range(DIFF_TOTAL_STEPS):
    x, cond = next(graph_only_iter)
    x = x.to(DEVICE)

    model_kwargs = {
        'adj_matrix': cond['adj_matrix'].to(DEVICE),
        'node_mask':  cond['node_mask'].to(DEVICE),
    }

    t = torch.randint(0, 1000, (x.shape[0],), device=DEVICE)

    with torch.amp.autocast('cuda', enabled=use_amp):
        loss, coord_rmse = diffusion.training_losses(diff_model_go, x, t, model_kwargs)

    diff_opt_go.zero_grad()
    scaler_go.scale(loss).backward()
    scaler_go.unscale_(diff_opt_go)
    nn.utils.clip_grad_norm_(diff_model_go.parameters(), 1.0)
    scaler_go.step(diff_opt_go)
    scaler_go.update()

    running_loss += loss.item()
    running_rmse += coord_rmse

    if step % DIFF_LOG == 0 and step > 0:
        print(f'step {step:6d} | loss {running_loss/DIFF_LOG:.4f} | '
              f'coord_rmse {running_rmse/DIFF_LOG:.2f} px | '
              f'scale {scaler_go.get_scale():.0f}')
        running_loss = running_rmse = 0.0

    if step > 0 and step % DIFF_SAVE_EVERY == 0:
        path = f'{DIFF_SAVE}/model_{step:07d}.pt'
        torch.save({'model': diff_model_go.state_dict(),
                    'opt': diff_opt_go.state_dict(), 'step': step}, path)
        print(f'  saved -> {path}')

# final save
final_path = f'{DIFF_SAVE}/model_final.pt'
torch.save({'model': diff_model_go.state_dict(),
            'opt': diff_opt_go.state_dict(), 'step': DIFF_TOTAL_STEPS}, final_path)
print(f'训练完成 -> {final_path}')

## 5. 文本编码 + 图结构条件全量扩散训练（text+graph）

加载 graph_only 预训练权重，接入冻结的 BERT 编码 prompt，文本 hidden states 通过 cross-attention 注入每一层。

In [ ]:
from transformers import BertModel

TG_TOTAL_STEPS = 200000
TG_LR          = 5e-5
TG_SAVE        = 'checkpoints/node_diffusion_text_graph'
os.makedirs(TG_SAVE, exist_ok=True)

# text+graph 数据集与 graph_only 相同（NodeTextDataset 同时含坐标和文本字段）
tg_loader = DataLoader(graph_only_ds, batch_size=DIFF_BATCH,
                       shuffle=True, num_workers=0, drop_last=True)
tg_iter   = infinite(tg_loader)

# 扩散模型：加载 graph_only final 权重
diff_model_tg = NodeDiffusionTransformer(
    model_channels=256, num_layers=6, num_heads=4
).to(DEVICE)

graph_only_ckpt = f'{DIFF_SAVE}/model_final.pt'   # final save 路径
if os.path.exists(graph_only_ckpt):
    ckpt = torch.load(graph_only_ckpt, map_location=DEVICE)
    diff_model_tg.load_state_dict(ckpt['model'])
    print(f'加载 graph_only 权重: {graph_only_ckpt}')
else:
    print('未找到预训练权重，从随机初始化开始')

# BERT 编码器：从 SFT checkpoint 提取微调后的 BERT 权重
bert_encoder = BertModel.from_pretrained(BERT_PATH).to(DEVICE)
sft_ckpt_path = f'{SFT_SAVE}/best.pt'
if os.path.exists(sft_ckpt_path):
    sft_state  = torch.load(sft_ckpt_path, map_location=DEVICE)
    bert_state = {k[len('bert.'):]: v for k, v in sft_state.items() if k.startswith('bert.')}
    missing, unexpected = bert_encoder.load_state_dict(bert_state, strict=False)
    print(f'从 SFT checkpoint 加载 BERT 权重: {sft_ckpt_path}')
    print(f'  missing={len(missing)}  unexpected={len(unexpected)}')
else:
    print(f'未找到 SFT checkpoint ({sft_ckpt_path})，使用原始 BERT 权重')

bert_encoder.eval()
for p in bert_encoder.parameters():
    p.requires_grad = False

diff_opt_tg = AdamW(diff_model_tg.parameters(), lr=TG_LR, weight_decay=1e-4)
print('text+graph 训练准备完成')

In [ ]:
diff_model_tg.train()
running_loss  = running_rmse = 0.0
scaler_tg     = torch.amp.GradScaler('cuda', enabled=use_amp)

for step in range(TG_TOTAL_STEPS):
    x, cond = next(tg_iter)
    x = x.to(DEVICE)

    with torch.no_grad():
        bert_out = bert_encoder(
            input_ids=cond['text_ids'].to(DEVICE),
            attention_mask=cond['text_mask'].to(DEVICE),
        )
        encoder_hidden = bert_out.last_hidden_state   # [B, L, 768]

    model_kwargs = {
        'adj_matrix':     cond['adj_matrix'].to(DEVICE),
        'node_mask':      cond['node_mask'].to(DEVICE),
        'encoder_hidden': encoder_hidden,
    }

    t = torch.randint(0, 1000, (x.shape[0],), device=DEVICE)

    with torch.amp.autocast('cuda', enabled=use_amp):
        loss, coord_rmse = diffusion.training_losses(diff_model_tg, x, t, model_kwargs)

    diff_opt_tg.zero_grad()
    scaler_tg.scale(loss).backward()
    scaler_tg.unscale_(diff_opt_tg)
    nn.utils.clip_grad_norm_(diff_model_tg.parameters(), 1.0)
    scaler_tg.step(diff_opt_tg)
    scaler_tg.update()

    running_loss += loss.item()
    running_rmse += coord_rmse

    if step % DIFF_LOG == 0 and step > 0:
        print(f'step {step:6d} | loss {running_loss/DIFF_LOG:.4f} | '
              f'coord_rmse {running_rmse/DIFF_LOG:.2f} px | '
              f'scale {scaler_tg.get_scale():.0f}')
        running_loss = running_rmse = 0.0

    if step > 0 and step % DIFF_SAVE_EVERY == 0:
        path = f'{TG_SAVE}/model_{step:07d}.pt'
        torch.save({'model': diff_model_tg.state_dict(),
                    'opt': diff_opt_tg.state_dict(), 'step': step}, path)
        print(f'  saved -> {path}')

# final save
final_path = f'{TG_SAVE}/model_final.pt'
torch.save({'model': diff_model_tg.state_dict(),
            'opt': diff_opt_tg.state_dict(), 'step': TG_TOTAL_STEPS}, final_path)
print(f'训练完成 -> {final_path}')